<a href="https://colab.research.google.com/github/ahmad1bakundi-ops/data-engineering-journey/blob/main/ngn-fx-project/ngn_fx_project_02_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import yfinance as yf

# Pull the data fresh
tickers = ['USDNGN=X', 'EURNGN=X', 'GBPNGN=X']  # no JPY, we know it fails
data = yf.download(
    tickers,
    start='2015-01-01',
    end='2025-01-01',
    interval='1d',
    progress=False,
    auto_adjust=True
)

# Simplify to just closing prices
close = data['Close'].copy()

# Rename to friendly names (alphabetical order matches columns)
close.columns = ['EUR', 'GBP', 'USD']

close.shape

(2607, 3)

In [26]:
# --- Cleaning pass 1: obvious bad ticks ---
close.loc[close['USD'] < 100, 'USD'] = np.nan
close.loc[close['GBP'] < 250, 'GBP'] = np.nan
close = close.interpolate(method='linear')

# --- Cleaning pass 2: cross-column relationship check ---
# GBP must always be > USD historically
close.loc[close['GBP'] <= close['USD'], 'GBP'] = np.nan
close = close.interpolate(method='linear')

# Verify all clean
close.isnull().sum(), close.describe()

(EUR    0
 GBP    0
 USD    0
 dtype: int64,
                EUR          GBP          USD
 count  2607.000000  2607.000000  2607.000000
 mean    534.403472   628.678642   482.317704
 std     387.965825   456.703779   361.348080
 min     206.529999   269.850006   178.610001
 25%     356.039993   425.921844   337.035004
 50%     423.559998   483.679993   363.269989
 75%     481.570007   556.980743   415.390015
 max    1864.699951  2232.942871  1696.199951)

In [27]:
close.to_csv(PROJECT_PATH + 'ngn_fx_daily_clean.csv')

In [28]:
fx = close
fx.head()

,EUR,GBP,USD
Date,,,
2015-01-01,219.429993,284.239990,182.300003
2015-01-02,219.429993,286.420013,181.380005
2015-01-05,218.139999,279.579987,182.699997
2015-01-06,216.110001,279.869995,183.199997
2015-01-07,216.389999,278.179993,183.600006


In [29]:
monthly = fx.resample('ME').mean()
monthly.head(10)

,EUR,GBP,USD
Date,,,
2015-01-31,214.397272,282.282272,185.568638
2015-02-28,222.147998,302.681998,197.016500
2015-03-31,213.754544,299.341819,199.338182
2015-04-30,211.163182,297.109546,198.195456
2015-05-31,219.155236,307.826669,198.659998
2015-06-30,219.918181,309.570457,198.732270
2015-07-31,216.226086,309.637390,198.693043
2015-08-31,218.158096,310.257144,197.580952
2015-09-30,220.445000,305.414096,198.439544


In [30]:
yearly = fx.resample('YE').mean()
yearly

,EUR,GBP,USD
Date,,,
2015-12-31,216.658352,302.401706,197.015555
2016-12-31,282.390920,351.906361,256.667625
2017-12-31,373.612791,436.025621,334.503062
2018-12-31,422.506973,482.164158,359.754597
2019-12-31,402.042662,459.691554,358.292510
2020-12-31,432.190450,485.870449,376.648232
2021-12-31,481.197663,551.154661,399.313738
2022-12-31,443.692770,522.947689,422.599922
2023-12-31,682.890612,794.570137,635.211950


In [31]:
yoy = yearly.pct_change() * 100
yoy.round(1)

,EUR,GBP,USD
Date,,,
2015-12-31,NaN,NaN,NaN
2016-12-31,30.3,16.4,30.3
2017-12-31,32.3,23.9,30.3
2018-12-31,13.1,10.6,7.5
2019-12-31,-4.8,-4.7,-0.4
2020-12-31,7.5,5.7,5.1
2021-12-31,11.3,13.4,6.0
2022-12-31,-7.8,-5.1,5.8
2023-12-31,53.9,51.9,50.3


In [32]:
yoy['spread'] = yoy.max(axis=1) - yoy.min(axis=1)
yoy.round(1)

,EUR,GBP,USD,spread
Date,,,,
2015-12-31,NaN,NaN,NaN,NaN
2016-12-31,30.3,16.4,30.3,14.0
2017-12-31,32.3,23.9,30.3,8.4
2018-12-31,13.1,10.6,7.5,5.5
2019-12-31,-4.8,-4.7,-0.4,4.4
2020-12-31,7.5,5.7,5.1,2.4
2021-12-31,11.3,13.4,6.0,7.4
2022-12-31,-7.8,-5.1,5.8,13.6
2023-12-31,53.9,51.9,50.3,3.6


In [33]:
crisis_period = monthly.loc['2023-01-01':'2024-12-31']
crisis_period.round(2)

,EUR,GBP,USD
Date,,,
2023-01-31,487.61,554.59,452.96
2023-02-28,490.80,556.71,459.61
2023-03-31,489.32,558.99,458.68
2023-04-30,501.68,574.96,460.80
2023-05-31,499.10,575.95,460.02
2023-06-30,643.64,772.05,602.12
2023-07-31,858.10,1004.23,779.31
2023-08-31,833.50,973.43,765.33
2023-09-30,820.93,957.59,771.47


In [34]:
mom = crisis_period.pct_change() * 100
mom.round(2)

,EUR,GBP,USD
Date,,,
2023-01-31,NaN,NaN,NaN
2023-02-28,0.65,0.38,1.47
2023-03-31,-0.30,0.41,-0.20
2023-04-30,2.53,2.86,0.46
2023-05-31,-0.52,0.17,-0.17
2023-06-30,28.96,34.05,30.89
2023-07-31,33.32,30.07,29.43
2023-08-31,-2.87,-3.07,-1.79
2023-09-30,-1.51,-1.63,0.80


In [35]:
yoy.to_csv(PROJECT_PATH + 'ngn_fx_yearly_analysis.csv')
mom.to_csv(PROJECT_PATH + 'ngn_fx_2023_2024_monthly.csv')